# 1. ollama  설치

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
!pkill -9 ollama

import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 29070 )


In [3]:
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text

# 2. agent 정의

- from langgraph.prebuilt import create_react_agent 를 사용

In [4]:
!pip install -q --upgrade \
  langchain langchain-core langchain-community langchain-classic langchain_chroma \
  langchain-huggingface langchain-ollama \
  chromadb sentence-transformers \
  opentelemetry-api opentelemetry-sdk \
  opentelemetry-exporter-otlp-proto-common opentelemetry-exporter-otlp-proto-grpc

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.43.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.43.0 which is incomp

In [5]:
!pip install langchain-text-splitters -q
!pip install langgraph -q
!pip install pypdf -q
!pip install pdfplumber -q

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# 1. LLM 설정
llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)


# 2. Tool 정의
@tool
def add_numbers(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    return a + b


@tool
def check_machine_error(symptom: str) -> str:
    """제조 설비 증상에 따른 원인을 추정한다."""
    if "burr" in symptom.lower() or "버" in symptom:
        return "버 발생은 공구 마모, 절삭 조건 불량, 스핀들 진동 증가를 먼저 점검해야 합니다."
    elif "치수" in symptom:
        return "치수 불량은 공구 마모, 지그 고정 불량, 온도 변화 가능성을 점검해야 합니다."
    else:
        return "증상이 명확하지 않습니다. 설비명, 불량 유형, 발생 시점을 추가로 확인해야 합니다."


tools = [add_numbers, check_machine_error]

In [7]:
# Agent 생성
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=(
        "당신은 제조업 엔진 진단 전문가입니다.\n"
        "- 사내 보고서 관련 질문 → search_reports 사용\n"
        "- 최신 정보/외부 기술 질문 → search_internet 사용\n"
        "- 필요시 두 Tool을 모두 사용해 종합 답변하세요."
    ),
)

/tmp/ipykernel_28189/804368053.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [8]:
# 4. 실행
result = agent.invoke({
    "messages": [
        {"role":"user", "content": "CNC에서 버가 계속 발생해. 원인이 뭘까?"},
        {"role":"user", "content": "3+5는 뭘까?"}
    ]
})

In [9]:
print( result["messages"][-1].content )


제조 설비의 증상에 따른 원인을 추정해보니, 버가 계속 발생하는 것은 공구 마모, 절삭 조건 불량, 또는 스핀들 진동 증가로 인한 것 같습니다. 

그러나 이는 CNC에서 버가 발생할 수 있는 몇 가지 일반적인 원인 중 하나일 뿐입니다. 더 정확한 진단을 위해서는 추가적인 검사와 분석이 필요합니다.

더불어 3+5는 8입니다.


# 3. 다양한 tool 정의

### 1) Chroma DB RAG 설정
- Chroma DB 생성

In [10]:
!pip install -q \
  opentelemetry-sdk==1.27.0 \
  opentelemetry-api==1.27.0 \
  opentelemetry-exporter-otlp-proto-grpc==1.27.0 \
  chromadb \
  langchain-chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.27.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.27.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.27.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but 

In [11]:
!rm -r chroma_db

In [12]:
!mkdir -p chroma_db 

In [14]:
#del vectorstore

In [15]:
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings

vectorstore = Chroma(
    collection_name="factory_docs",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text"),
    persist_directory="./chroma_db"  # 디스크에 저장
)

# 문서 추가
#vectorstore.add_documents(문자열 리스트, ids=[문서의 개수와 동일한 아이디 문자열])

/tmp/ipykernel_28189/3124549173.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings
/tmp/ipykernel_28189/3124549173.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_function=OllamaEmbeddings(model="nomic-embed-text"),


- Chroma DB 조회

In [16]:
# 몇 개 들어있는지
print(vectorstore._collection.count())

# 전체 조회
vectorstore.get()

0


{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

- Chroma DB에 문서 추가

In [17]:
from langchain_ollama import ChatOllama
from langchain_core.documents import Document          
from langchain_classic.chains import RetrievalQA       
import json

In [18]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-29 09:59:12--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861 (124K) [text/plain]
Saving to: ‘reports.json.4’

reports.json.4      100%[===================>] 123.89K  --.-KB/s    in 0.001s  

2026-06-29 09:59:12 (80.8 MB/s) - ‘reports.json.4’ saved [126861/126861]



In [19]:
# --- reports.json 로드 ---
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)
reports = reports[:10]

#lc_embeddings = OllamaEmbeddings(model="nomic-embed-text")

lc_documents = [
    Document(
        page_content=r["report_text"],
        metadata={"report_id": r["report_id"], "source": "json"}
    )
    for r in reports
]

In [20]:
#Chroma DB에 추가
vectorstore.add_documents(lc_documents)
print(f"추가 완료: 총 {vectorstore._collection.count()}개")

추가 완료: 총 10개


- Chroma DB에 문서 추가: pdf

In [21]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf

--2026-06-29 09:59:14--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1432218 (1.4M) [application/octet-stream]
Saving to: ‘manual.pdf.2’

manual.pdf.2        100%[===================>]   1.37M  --.-KB/s    in 0.01s   

2026-06-29 09:59:14 (119 MB/s) - ‘manual.pdf.2’ saved [1432218/1432218]



In [22]:
#PDF 로드 및 청킹
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_loader = PyPDFLoader("manual.pdf")
pdf_pages  = pdf_loader.load()          # 페이지별 Document 리스트

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pdf_docs = splitter.split_documents(pdf_pages)

# source 메타데이터 부여
for doc in pdf_docs:
    doc.metadata["source"] = "pdf"

print(f"PDF 청크 수: {len(pdf_docs)}")

PDF 청크 수: 14


In [23]:
#Chroma DB에 추가
vectorstore.add_documents(pdf_docs)
print(f"추가 완료: 총 {vectorstore._collection.count()}개")

추가 완료: 총 24개


- Vector Store 검색 tool 정의

In [24]:
# 벡터스토어 검색 리트리버 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

@tool
def search_engine_reports(query: str) -> str:
    """엔진 고장 및 불량 보고서에서 관련 정보를 검색합니다. 제조 불량 원인 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾지 못했습니다."

    results = []
    for i, doc in enumerate(docs, 1):
        src    = doc.metadata.get("source", "unknown")
        ref_id = doc.metadata.get("report_id", doc.metadata.get("page", ""))
        snippet = doc.page_content[:300]
        results.append(f"[{i}] 출처={src} | ID/페이지={ref_id}\n{snippet}")

    return "\n\n".join(results)

tools.append(search_engine_reports)

### 2) tool 추가: 인터넷 검색

In [25]:
!pip install ddgs -q

In [26]:
from langchain_community.tools import DuckDuckGoSearchRun
duckduckgo = DuckDuckGoSearchRun()

@tool
def search_internet(query: str) -> str:
    """인터넷에서 최신 정보를 검색합니다. 최신 뉴스, 외부 기술 정보, 일반 지식 질문에 사용하세요."""
    return duckduckgo.invoke(query)


tools.append(search_internet)

### 3) tool을 추가: DB조회

In [27]:
!pip install duckdb

In [28]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/work_logs.csv

--2026-06-29 09:59:25--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/work_logs.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4239 (4.1K) [text/plain]
Saving to: ‘work_logs.csv.2’

work_logs.csv.2     100%[===================>]   4.14K  --.-KB/s    in 0s      

2026-06-29 09:59:25 (75.6 MB/s) - ‘work_logs.csv.2’ saved [4239/4239]



In [29]:
import duckdb
import pandas as pd

# CSV 로드
df = pd.read_csv("work_logs.csv", encoding="utf-8-sig")
print(f"로드 완료: {len(df)}건")
print(df.head(3))

# DuckDB에 저장
con = duckdb.connect("factory.db")
con.execute("DROP TABLE IF EXISTS work_logs")
con.execute("CREATE TABLE work_logs AS SELECT * FROM df")
print(f"\nwork_logs 테이블 생성: {con.execute('SELECT COUNT(*) FROM work_logs').fetchone()[0]}건")
con.close()

로드 완료: 30건
   log_id              작업일시         설비명  작업유형  작업자  소요시간_h 작업결과    비용_원  \
0  WL-001  2024-06-14 00:00  CNC 머신 A-2  정기점검  박민준     2.3   완료   31000   
1  WL-002  2024-06-24 00:00  레이저 커터 L-1  정기점검  정수호     3.7   완료   28000   
2  WL-003  2024-02-27 00:00    용접로봇 W-1  필터청소  정수호     0.7   완료  144000   

                        비고  
0  CNC 머신 A-2 정기점검 수행 - 완료  
1  레이저 커터 L-1 정기점검 수행 - 완료  
2    용접로봇 W-1 필터청소 수행 - 완료  

work_logs 테이블 생성: 30건


In [30]:
con = duckdb.connect("factory.db")
result = con.execute("SELECT * FROM work_logs LIMIT 5").df()
print(result.to_string())
con.close()

   log_id              작업일시         설비명  작업유형  작업자  소요시간_h 작업결과    비용_원                       비고
0  WL-001  2024-06-14 00:00  CNC 머신 A-2  정기점검  박민준     2.3   완료   31000  CNC 머신 A-2 정기점검 수행 - 완료
1  WL-002  2024-06-24 00:00  레이저 커터 L-1  정기점검  정수호     3.7   완료   28000  레이저 커터 L-1 정기점검 수행 - 완료
2  WL-003  2024-02-27 00:00    용접로봇 W-1  필터청소  정수호     0.7   완료  144000    용접로봇 W-1 필터청소 수행 - 완료
3  WL-004  2024-04-19 00:00    용접로봇 W-1  오일교환  정수호     2.6   완료   45000    용접로봇 W-1 오일교환 수행 - 완료
4  WL-005  2024-06-29 00:00     프레스 P-2  부품교체  박민준     1.7   완료   31000     프레스 P-2 부품교체 수행 - 완료


In [31]:
@tool
def query_work_logs(query: str) -> str:
    """
    공장 작업로그 DB를 SQL로 조회합니다.
    설비별 작업이력, 작업자, 소요시간, 비용, 작업결과를 조회할 수 있습니다.
    테이블명: work_logs
    컬럼: log_id, 작업일시, 설비명, 작업유형, 작업자, 소요시간_h, 작업결과, 비용_원, 비고
    """
    try:
        con = duckdb.connect("factory.db", read_only=True)
        result = con.execute(query).df()
        con.close()
        if result.empty:
            return "조회 결과가 없습니다."
        return result.to_string(index=False)
    except Exception as e:
        return f"쿼리 오류: {str(e)}"
    
tools.append(query_work_logs)

In [32]:
# Tool 직접 호출 테스트 (Agent 없이)
print(query_work_logs.invoke(
    "SELECT 설비명, COUNT(*) as 작업수, SUM(비용_원) as 총비용 FROM work_logs GROUP BY 설비명 ORDER BY 총비용 DESC"
))

       설비명  작업수      총비용
  용접로봇 W-1    5 400000.0
레이저 커터 L-1    6 388000.0
   프레스 P-2    4 282000.0
  용접로봇 W-2    3 282000.0
   프레스 P-1    4 221000.0
  컨베이어 C-1    4 137000.0
CNC 머신 A-1    1 129000.0
CNC 머신 B-1    1 102000.0
CNC 머신 A-2    2  95000.0


### 4) ML호출

- 간단한  ML 모형

In [33]:
import pandas as pd
import duckdb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

# DB에서 학습 데이터 로드
con = duckdb.connect("factory.db", read_only=True)
df = con.execute("SELECT * FROM work_logs").df()
con.close()

# 피처 인코딩
le_fac = LabelEncoder()
le_work = LabelEncoder()

df["facility_enc"] = le_fac.fit_transform(df["설비명"])
df["work_enc"] = le_work.fit_transform(df["작업유형"])

# 불량 여부 (작업결과 != "완료" → 1)
df["defect"] = (df["작업결과"] != "완료").astype(int)

X = df[["facility_enc", "work_enc", "소요시간_h", "비용_원"]]
y = df["defect"]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# 모델 + 인코더 저장
with open("defect_model.pkl", "wb") as f:
    pickle.dump({"model": model, "le_설비": le_fac, "le_작업": le_work}, f)

print("모델 학습 완료")
print(f"   학습 데이터: {len(df)}건 / 불량 비율: {y.mean():.1%}")

모델 학습 완료
   학습 데이터: 30건 / 불량 비율: 30.0%


- tool 정의

In [34]:
import pickle

# 모델 로드 (전역)
with open("defect_model.pkl", "rb") as f:
    artifacts = pickle.load(f)
    clf = artifacts["model"]
    le_설비 = artifacts["le_설비"]
    le_작업 = artifacts["le_작업"]

@tool
def predict_defect(설비명: str, 작업유형: str, 소요시간_h: float, 비용_원: int) -> str:
    """
    설비 작업 정보를 입력받아 불량 발생 가능성을 예측합니다.
    DB 조회 결과에서 얻은 값을 그대로 입력하세요.
    입력: 설비명, 작업유형, 소요시간_h, 비용_원
    출력: 불량 예측 결과 및 확률
    """
    try:
        # 인코딩
        설비_enc = le_설비.transform([설비명])[0]
        작업_enc = le_작업.transform([작업유형])[0]

        X = [[설비_enc, 작업_enc, 소요시간_h, 비용_원]]
        pred = clf.predict(X)[0]
        prob = clf.predict_proba(X)[0][1]

        result = "불량 가능성 높음" if pred == 1 else "정상"
        return f"{result} | 불량 확률: {prob:.1%} | 설비: {설비명} | 작업: {작업유형}"

    except ValueError as e:
        return f"예측 불가 (미학습 값 포함): {str(e)}"
    
tools.append( predict_defect)

In [35]:
# 직접 호출 테스트
print(predict_defect.invoke({
    "설비명": "프레스 P-2",
    "작업유형": "정기점검",
    "소요시간_h": 6.9,
    "비용_원": 72000
}))

불량 가능성 높음 | 불량 확률: 78.0% | 설비: 프레스 P-2 | 작업: 정기점검


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


### 5) 실행

In [36]:
from langgraph.prebuilt import create_react_agent
# LLM 설정 및 agent  정의
llm2 = ChatOllama( model="qwen2.5:3b", temperature=0)
agent2 = create_react_agent(
    model=llm2,
    tools=tools,
    prompt=(
        "당신은 제조업 엔진 진단 전문가입니다.\n"
        "- 사내 보고서 관련 질문 → search_reports 사용\n"
        "- 최신 정보/외부 기술 질문 → search_internet 사용\n"
        "- 필요시 두 Tool을 모두 사용해 종합 답변하세요."
    ),
)


/tmp/ipykernel_28189/1414353089.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent2 = create_react_agent(


In [56]:
# 실행
result2 = agent2.invoke({
    "messages": [
#       {"role":"user", "content": "CNC에서 버가 계속 발생해. 원인이 뭘까?"},
#       {"role":"user", "content": "today it news"},
#        {"role":"user", "content": "설비명: 프레스 P-2, 작업유형: 정기점검, 소요시간_h: 6.9, 비용_원: 72000으로 predict defect"},
        {"role":"user", "content": "execute query_work_logs"},    
    ]
})

In [57]:
print( result2["messages"][-1].content )

The query worked successfully and returned a log entry from the `work_logs` table. This data includes various details such as `log_id`, `작업일시` (operation date), `설비명` (equipment name), `작업유형` (type of operation), `작업자` (operator), `소요시간_h` (hours worked), `작업결과` (outcome), and `비용_원` (cost in 원).

If you need to fetch more data or perform a different query, please let me know.
